# 💰 GUI Expense Tracker (GUI)

A full-featured expense manager with a **point-and-click GUI** built using `ipywidgets`, a **SQLite** database, **pandas** analytics and interactive **Plotly** dashboards.

**Features**
- ➕ Add / ✏️ edit / 🗑️ delete expenses
- 🔍 Filter by date, category, note search
- 📊 Interactive dashboard (category split, monthly trend, cumulative spend, payment methods)
- 🎯 Monthly budgets with colour-coded progress bars and alerts
- 💡 Smart insights (month-end projection, weekday vs weekend, top categories)
- 📁 Import/Export CSV & Excel, sample data generator, optional Google Drive persistence



## 1. Setup & configuration

In [ ]:
import sqlite3, io, random
from datetime import date, datetime, timedelta

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as w
from IPython.display import display, clear_output, HTML

pio.renderers.default = "colab"

# ---- Settings ---------------------------------------------------------------
USE_DRIVE = True          # True = save the database in Google Drive so data survives restarts
CURRENCY = "₹"
CATEGORIES = ["Food", "Transport", "Shopping", "Bills", "Entertainment",
              "Health", "Education", "Travel", "Other"]
METHODS = ["Cash", "UPI", "Credit Card", "Debit Card", "Net Banking"]
# -----------------------------------------------------------------------------

DB_PATH = "expense_tracker.db"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DB_PATH = "/content/drive/MyDrive/expense_tracker.db"
        print("✅ Using Google Drive database:", DB_PATH)
    except Exception as e:
        print("⚠️ Drive not available, using local file instead:", e)

def money(x):
    return f"{CURRENCY}{x:,.2f}"

## 2. Database layer (SQLite)

In [ ]:
class ExpenseDB:
    def __init__(self, path):
        self.conn = sqlite3.connect(path, check_same_thread=False)
        self.conn.execute("""CREATE TABLE IF NOT EXISTS expenses(
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            date TEXT NOT NULL,
            category TEXT NOT NULL,
            amount REAL NOT NULL CHECK(amount > 0),
            method TEXT,
            note TEXT)""")
        self.conn.execute("""CREATE TABLE IF NOT EXISTS budgets(
            category TEXT PRIMARY KEY,
            monthly_limit REAL NOT NULL)""")
        self.conn.commit()

    def add(self, d, cat, amt, method, note):
        self.conn.execute("INSERT INTO expenses(date,category,amount,method,note) VALUES(?,?,?,?,?)",
                          (d, cat, float(amt), method, note))
        self.conn.commit()

    def add_many(self, rows):
        self.conn.executemany("INSERT INTO expenses(date,category,amount,method,note) VALUES(?,?,?,?,?)", rows)
        self.conn.commit()

    def update(self, id_, d, cat, amt, method, note):
        self.conn.execute("UPDATE expenses SET date=?,category=?,amount=?,method=?,note=? WHERE id=?",
                          (d, cat, float(amt), method, note, id_))
        self.conn.commit()

    def delete(self, id_):
        self.conn.execute("DELETE FROM expenses WHERE id=?", (id_,))
        self.conn.commit()

    def clear(self):
        self.conn.execute("DELETE FROM expenses")
        self.conn.commit()

    def get(self, id_):
        return self.conn.execute("SELECT id,date,category,amount,method,note FROM expenses WHERE id=?",
                                 (id_,)).fetchone()

    def df(self, start=None, end=None, category=None):
        q, p = "SELECT id,date,category,amount,method,note FROM expenses WHERE 1=1", []
        if start:
            q += " AND date>=?"; p.append(start.isoformat())
        if end:
            q += " AND date<=?"; p.append(end.isoformat())
        if category:
            q += " AND category=?"; p.append(category)
        q += " ORDER BY date DESC, id DESC"
        return pd.read_sql_query(q, self.conn, params=p)

    def set_budget(self, cat, limit):
        if limit > 0:
            self.conn.execute("INSERT OR REPLACE INTO budgets(category,monthly_limit) VALUES(?,?)", (cat, float(limit)))
        else:
            self.conn.execute("DELETE FROM budgets WHERE category=?", (cat,))
        self.conn.commit()

    def budgets(self):
        return dict(self.conn.execute("SELECT category,monthly_limit FROM budgets").fetchall())


db = ExpenseDB(DB_PATH)

def load_df():
    """All expenses with a proper datetime column (used by analytics)."""
    df = db.df()
    df["date"] = pd.to_datetime(df["date"])
    return df

## 3. GUI — Tab 1: Add expense

In [ ]:
FIELD = {"description_width": "110px"}
WIDE = w.Layout(width="420px")

a_date = w.DatePicker(description="Date", value=date.today(), style=FIELD, layout=WIDE)
a_cat = w.Dropdown(description="Category", options=CATEGORIES, style=FIELD, layout=WIDE)
a_amt = w.FloatText(description=f"Amount ({CURRENCY})", value=0.0, style=FIELD, layout=WIDE)
a_method = w.Dropdown(description="Payment", options=METHODS, style=FIELD, layout=WIDE)
a_note = w.Text(description="Note", placeholder="e.g. Lunch with friends", style=FIELD, layout=WIDE)
a_btn = w.Button(description="Add Expense", icon="plus", button_style="success")
a_out = w.Output()

def on_add(_):
    with a_out:
        clear_output()
        if a_date.value is None:
            print("⚠️ Please pick a date."); return
        if a_amt.value <= 0:
            print("⚠️ Amount must be greater than 0."); return
        db.add(a_date.value.isoformat(), a_cat.value, a_amt.value, a_method.value, a_note.value.strip())
        print(f"✅ Added {money(a_amt.value)} under {a_cat.value}")
        a_amt.value = 0.0
        a_note.value = ""
    refresh_all()

a_btn.on_click(on_add)
tab_add = w.VBox([w.HTML("<h3>➕ Add a new expense</h3>"), a_date, a_cat, a_amt, a_method, a_note, a_btn, a_out])

## 4. GUI — Tab 2: Records (filter, edit, delete)

In [ ]:
r_from = w.DatePicker(description="From", style={"description_width": "60px"})
r_to = w.DatePicker(description="To", style={"description_width": "60px"})
r_cat = w.Dropdown(description="Category", options=["All"] + CATEGORIES, style={"description_width": "80px"})
r_search = w.Text(description="Search", placeholder="note contains...", style={"description_width": "60px"})
r_out = w.Output()

# edit panel
e_sel = w.Dropdown(description="Record", options=[], style=FIELD, layout=w.Layout(width="520px"))
e_date = w.DatePicker(description="Date", style=FIELD, layout=WIDE)
e_cat = w.Dropdown(description="Category", options=CATEGORIES, style=FIELD, layout=WIDE)
e_amt = w.FloatText(description=f"Amount ({CURRENCY})", style=FIELD, layout=WIDE)
e_method = w.Dropdown(description="Payment", options=METHODS, style=FIELD, layout=WIDE)
e_note = w.Text(description="Note", style=FIELD, layout=WIDE)
e_update = w.Button(description="Update", icon="save", button_style="info")
e_delete = w.Button(description="Delete", icon="trash", button_style="danger")
e_msg = w.Output()

def load_edit(_=None):
    if e_sel.value is None:
        return
    row = db.get(e_sel.value)
    if row:
        e_date.value = date.fromisoformat(row[1])
        e_cat.value = row[2] if row[2] in CATEGORIES else "Other"
        e_amt.value = row[3]
        e_method.value = row[4] if row[4] in METHODS else METHODS[0]
        e_note.value = row[5] or ""

e_sel.observe(load_edit, names="value")

def refresh_records(_=None):
    df = db.df(r_from.value, r_to.value, None if r_cat.value == "All" else r_cat.value)
    if r_search.value:
        df = df[df["note"].fillna("").str.contains(r_search.value, case=False, regex=False)]
    with r_out:
        clear_output(wait=True)
        if df.empty:
            print("No records found.")
        else:
            show = df.head(200).rename(columns=str.title)
            display(show.style.format({"Amount": CURRENCY + "{:,.2f}"}).hide(axis="index"))
            extra = f" (showing first 200)" if len(df) > 200 else ""
            print(f"{len(df)} records{extra} • total {money(df.amount.sum())}")
    e_sel.options = [(f"#{r.id} | {r.date} | {r.category} | {CURRENCY}{r.amount:,.0f} | {r.note or ''}", int(r.id))
                     for r in df.head(200).itertuples()]

def on_update(_):
    with e_msg:
        clear_output()
        if e_sel.value is None:
            print("⚠️ Select a record first."); return
        if e_amt.value <= 0 or e_date.value is None:
            print("⚠️ Enter a valid date and amount."); return
        db.update(e_sel.value, e_date.value.isoformat(), e_cat.value, e_amt.value, e_method.value, e_note.value.strip())
        print("✅ Record updated.")
    refresh_all()

def on_delete(_):
    with e_msg:
        clear_output()
        if e_sel.value is None:
            print("⚠️ Select a record first."); return
        db.delete(e_sel.value)
        print("🗑️ Record deleted.")
    refresh_all()

e_update.on_click(on_update)
e_delete.on_click(on_delete)
for wd in (r_from, r_to, r_cat, r_search):
    wd.observe(refresh_records, names="value")

tab_records = w.VBox([
    w.HTML("<h3>🔍 Browse, edit & delete</h3>"),
    w.HBox([r_from, r_to, r_cat, r_search]),
    r_out,
    w.HTML("<hr><h4>✏️ Edit or delete a record</h4>"),
    e_sel, e_date, e_cat, e_amt, e_method, e_note,
    w.HBox([e_update, e_delete]), e_msg,
])

## 5. GUI — Tab 3: Dashboard

In [ ]:
d_month = w.Dropdown(description="Period", options=["All time"], style={"description_width": "60px"})
d_out = w.Output()
_lock = False

def kpi(title, value, color):
    return (f"<div style='flex:1;min-width:160px;padding:14px;border-radius:12px;background:{color};"
            f"color:white;margin:4px'><div style='font-size:12px;opacity:.85'>{title}</div>"
            f"<div style='font-size:22px;font-weight:700'>{value}</div></div>")

def render_dashboard(_=None):
    if _lock:
        return
    df_all = load_df()
    with d_out:
        clear_output(wait=True)
        if df_all.empty:
            print("No data yet — add expenses or load sample data from the 'Data' tab.")
            return
        df = df_all
        if d_month.value != "All time":
            df = df_all[df_all["date"].dt.strftime("%Y-%m") == d_month.value]
        if df.empty:
            print("No expenses in this period."); return

        by_cat = df.groupby("category")["amount"].sum().sort_values(ascending=False)
        html = "<div style='display:flex;flex-wrap:wrap'>"
        html += kpi("Total spent", money(df.amount.sum()), "#4f46e5")
        html += kpi("Transactions", f"{len(df)}", "#0891b2")
        html += kpi("Average / transaction", money(df.amount.mean()), "#059669")
        html += kpi("Top category", f"{by_cat.index[0]}", "#d97706")
        html += "</div>"
        display(HTML(html))

        fig1 = px.pie(values=by_cat.values, names=by_cat.index, hole=0.45, title="Spending by category")
        fig1.update_layout(height=380, template="plotly_white")
        fig1.show()

        monthly = df_all.groupby(df_all["date"].dt.to_period("M").astype(str))["amount"].sum().reset_index()
        fig2 = px.bar(monthly, x="date", y="amount", title="Monthly spending trend",
                      labels={"date": "Month", "amount": f"Amount ({CURRENCY})"}, text_auto=".2s")
        fig2.update_layout(height=360, template="plotly_white")
        fig2.show()

        daily = df.groupby("date")["amount"].sum().sort_index().cumsum().reset_index()
        fig3 = px.area(daily, x="date", y="amount", title="Cumulative spending in selected period",
                       labels={"amount": f"Cumulative ({CURRENCY})", "date": "Date"})
        fig3.update_layout(height=340, template="plotly_white")
        fig3.show()

        by_pay = df.groupby("method")["amount"].sum().sort_values()
        fig4 = go.Figure(go.Bar(x=by_pay.values, y=by_pay.index, orientation="h"))
        fig4.update_layout(title="Payment methods", height=300, template="plotly_white")
        fig4.show()

def refresh_dashboard():
    global _lock
    df = load_df()
    months = sorted(df["date"].dt.strftime("%Y-%m").unique(), reverse=True) if not df.empty else []
    cur = d_month.value
    _lock = True
    d_month.options = ["All time"] + months
    d_month.value = cur if cur in d_month.options else "All time"
    _lock = False
    render_dashboard()

d_month.observe(render_dashboard, names="value")
tab_dash = w.VBox([w.HTML("<h3>📊 Dashboard</h3>"), d_month, d_out])

## 6. GUI — Tab 4: Budgets

In [ ]:
b_inputs = {c: w.FloatText(value=0.0, description=c, style={"description_width": "110px"},
                           layout=w.Layout(width="280px")) for c in CATEGORIES}
b_save = w.Button(description="Save budgets", icon="save", button_style="success")
b_msg = w.Output()
b_status = w.Output()

def load_budget_inputs():
    saved = db.budgets()
    for c, box in b_inputs.items():
        box.value = saved.get(c, 0.0)

def refresh_budget_status():
    budgets = db.budgets()
    df = load_df()
    month = date.today().strftime("%Y-%m")
    spent = df[df["date"].dt.strftime("%Y-%m") == month].groupby("category")["amount"].sum()
    with b_status:
        clear_output(wait=True)
        if not budgets:
            print("No budgets set. Enter monthly limits above (0 = no budget).")
            return
        html, alerts = f"<h4>📅 Budget status for {month}</h4>", []
        for c in CATEGORIES:
            lim = budgets.get(c, 0)
            if lim <= 0:
                continue
            s = float(spent.get(c, 0))
            pct = s / lim * 100
            color = "#16a34a" if pct < 80 else ("#f59e0b" if pct <= 100 else "#dc2626")
            if pct > 100:
                alerts.append(f"🚨 <b>{c}</b> is over budget by {money(s - lim)}")
            elif pct >= 80:
                alerts.append(f"⚠️ <b>{c}</b> has used {pct:.0f}% of its budget")
            html += (f"<div style='margin:8px 0;max-width:520px'><div style='display:flex;justify-content:space-between'>"
                     f"<span>{c}</span><span>{money(s)} / {money(lim)} ({pct:.0f}%)</span></div>"
                     f"<div style='background:#e5e7eb;border-radius:8px;height:14px'>"
                     f"<div style='width:{min(pct,100):.0f}%;background:{color};height:14px;border-radius:8px'></div></div></div>")
        if alerts:
            html = "<div style='padding:10px;border-left:4px solid #dc2626;background:#fef2f2;color:#111;max-width:520px'>" + \
                   "<br>".join(alerts) + "</div>" + html
        display(HTML(html))

def on_save_budget(_):
    for c, box in b_inputs.items():
        db.set_budget(c, box.value)
    with b_msg:
        clear_output()
        print("✅ Budgets saved.")
    refresh_budget_status()

b_save.on_click(on_save_budget)
tab_budget = w.VBox([
    w.HTML("<h3>🎯 Monthly budgets</h3>"),
    w.HBox([w.VBox(list(b_inputs.values())[:5]), w.VBox(list(b_inputs.values())[5:])]),
    b_save, b_msg, b_status,
])

## 7. GUI — Tab 5: Smart insights

In [ ]:
i_out = w.Output()

def refresh_insights():
    df = load_df()
    with i_out:
        clear_output(wait=True)
        if df.empty:
            print("Add some expenses to see insights."); return
        today = pd.Timestamp.today().normalize()
        period = df["date"].dt.to_period("M")
        this_m = df[period == today.to_period("M")]
        last_m = df[period == (today.to_period("M") - 1)]
        spent = this_m.amount.sum()
        projected = spent / today.day * today.days_in_month if today.day else 0
        items = []
        items.append(f"📆 <b>This month so far:</b> {money(spent)} (daily average {money(spent / today.day)})")
        items.append(f"🔮 <b>Projected month-end spend:</b> {money(projected)}")
        if last_m.amount.sum() > 0:
            change = (spent - last_m.amount.sum()) / last_m.amount.sum() * 100
            arrow = "📈 up" if change > 0 else "📉 down"
            items.append(f"{arrow} <b>{abs(change):.0f}%</b> compared with last month's total of {money(last_m.amount.sum())} "
                         f"(note: this month is not complete yet)")
        top3 = df.groupby("category")["amount"].sum().sort_values(ascending=False).head(3)
        items.append("🏆 <b>Top 3 categories (all time):</b> " + ", ".join(f"{k} ({money(v)})" for k, v in top3.items()))
        big = df.loc[df.amount.idxmax()]
        items.append(f"💸 <b>Largest single expense:</b> {money(big.amount)} on {big.category} ({big.date.date()})")
        per_day = df.groupby(df["date"].dt.date)["amount"].sum()
        per_day.index = pd.to_datetime(per_day.index)
        wk = per_day[per_day.index.dayofweek < 5].mean()
        we = per_day[per_day.index.dayofweek >= 5].mean()
        if pd.notna(wk) and pd.notna(we):
            items.append(f"🗓️ <b>Average spend on active days:</b> weekdays {money(wk)} vs weekends {money(we)}")
        best_day = per_day.groupby(per_day.index.day_name()).mean().sort_values(ascending=False)
        items.append(f"📌 <b>Highest-spending weekday on average:</b> {best_day.index[0]} ({money(best_day.iloc[0])})")
        display(HTML("<h3>💡 Smart insights</h3><ul style='line-height:1.9'>" +
                     "".join(f"<li>{i}</li>" for i in items) + "</ul>"))

tab_insights = w.VBox([i_out])

## 8. GUI — Tab 6: Data (import / export / demo data)

In [ ]:
x_csv = w.Button(description="Download CSV", icon="download")
x_xlsx = w.Button(description="Download Excel", icon="download")
x_upload = w.FileUpload(accept=".csv", multiple=False, description="Import CSV")
x_sample = w.Button(description="Load sample data", icon="database", button_style="warning")
x_confirm = w.Checkbox(description="I understand this deletes ALL expenses", indent=False)
x_clear = w.Button(description="Delete all data", icon="trash", button_style="danger")
x_out = w.Output()

def _download(path):
    try:
        from google.colab import files
        files.download(path)
    except Exception:
        print(f"Saved to {path}")

def on_csv(_):
    with x_out:
        clear_output()
        db.df().to_csv("expenses.csv", index=False)
        print("Preparing CSV...")
        _download("expenses.csv")

def on_xlsx(_):
    with x_out:
        clear_output()
        df = db.df()
        with pd.ExcelWriter("expenses.xlsx") as xw:
            df.to_excel(xw, sheet_name="Expenses", index=False)
            if not df.empty:
                df.groupby("category")["amount"].sum().reset_index().to_excel(xw, sheet_name="By Category", index=False)
        print("Preparing Excel file...")
        _download("expenses.xlsx")

def on_upload(change):
    val = x_upload.value
    files_ = val.values() if isinstance(val, dict) else val
    with x_out:
        clear_output()
        for f in files_:
            try:
                data = pd.read_csv(io.BytesIO(bytes(f["content"])))
                data.columns = [c.strip().lower() for c in data.columns]
                if not {"date", "category", "amount"} <= set(data.columns):
                    print("⚠️ CSV needs at least: date, category, amount (optional: method, note)"); continue
                rows = []
                for r in data.itertuples():
                    d = pd.to_datetime(r.date).date().isoformat()
                    cat = r.category if r.category in CATEGORIES else "Other"
                    method = getattr(r, "method", None)
                    method = method if method in METHODS else "Cash"
                    note = getattr(r, "note", "")
                    note = "" if pd.isna(note) else str(note)
                    if float(r.amount) > 0:
                        rows.append((d, cat, float(r.amount), method, note))
                db.add_many(rows)
                print(f"✅ Imported {len(rows)} rows.")
            except Exception as e:
                print("❌ Could not import file:", e)
    refresh_all()

def on_sample(_):
    random.seed(42)
    profile = {"Food": (80, 450, 4), "Transport": (30, 300, 3), "Shopping": (300, 3000, 1),
               "Bills": (500, 2500, 0.3), "Entertainment": (150, 1200, 1),
               "Health": (200, 1500, 0.3), "Education": (300, 2000, 0.3),
               "Travel": (500, 4000, 0.2), "Other": (50, 500, 1)}
    notes = {"Food": "Meals / groceries", "Transport": "Cab / metro / fuel", "Shopping": "Clothes / gadgets",
             "Bills": "Electricity / internet", "Entertainment": "Movies / OTT", "Health": "Pharmacy / doctor",
             "Education": "Books / courses", "Travel": "Trip", "Other": "Misc"}
    rows, start = [], date.today() - timedelta(days=120)
    for i in range(121):
        d = start + timedelta(days=i)
        for cat, (lo, hi, freq) in profile.items():
            if random.random() < freq / 7:
                rows.append((d.isoformat(), cat, round(random.uniform(lo, hi), 2),
                             random.choice(METHODS), notes[cat]))
    db.add_many(rows)
    with x_out:
        clear_output()
        print(f"✅ Added {len(rows)} sample expenses.")
    refresh_all()

def on_clear(_):
    with x_out:
        clear_output()
        if not x_confirm.value:
            print("⚠️ Tick the confirmation box first."); return
        db.clear()
        x_confirm.value = False
        print("🗑️ All expenses deleted.")
    refresh_all()

x_csv.on_click(on_csv); x_xlsx.on_click(on_xlsx); x_sample.on_click(on_sample); x_clear.on_click(on_clear)
x_upload.observe(on_upload, names="value")

tab_data = w.VBox([
    w.HTML("<h3>📁 Data tools</h3>"),
    w.HBox([x_csv, x_xlsx]),
    w.HTML("<b>Import CSV</b> (columns: date, category, amount, method, note)"),
    x_upload, x_sample,
    w.HTML("<hr>"), x_confirm, x_clear, x_out,
])

## 9. Launch the app 🚀

In [ ]:
def refresh_all():
    refresh_records()
    refresh_dashboard()
    refresh_budget_status()
    refresh_insights()

tabs = w.Tab(children=[tab_add, tab_records, tab_dash, tab_budget, tab_insights, tab_data])
for i, t in enumerate(["➕ Add", "🔍 Records", "📊 Dashboard", "🎯 Budgets", "💡 Insights", "📁 Data"]):
    tabs.set_title(i, t)

load_budget_inputs()
display(HTML("<h2 style='margin:4px 0'>💰 Smart Expense Tracker</h2>"
             "<div style='color:gray'>Track • Analyse • Stay within budget</div>"))
display(tabs)
refresh_all()